Read Bronze Delta Tables back into dataframes

In [0]:
customers_bronze_df = spark.table("workspace.bronze.customers")
products_bronze_df = spark.table("workspace.bronze.products")
orders_bronze_df = spark.table("workspace.bronze.orders")

In [0]:
display(customers_bronze_df)

In [0]:
customers_bronze_df.count()

In [0]:
customers_bronze_df.dropDuplicates().count()

In [0]:
customers_silver_df = customers_bronze_df.dropDuplicates()

In [0]:
from pyspark.sql.functions import col,sum
customers_silver_df.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in customers_silver_df.columns]
).show()

In [0]:
customers_silver_df.filter(col("country").isNull()).show()

In [0]:
customers_silver_df = customers_silver_df.fillna(
    {"country": "Unknown"}
)

In [0]:
customers_silver_df.filter(col("country").isNull()).show()

In [0]:
customers_silver_df.select("country").distinct().show()

In [0]:
from pyspark.sql.functions import col, initcap, trim

customers_silver_df = customers_silver_df.withColumn("country", initcap(trim(col("country"))))

In [0]:
customers_silver_df.select("country").distinct().show()

In [0]:

customers_silver_df.groupBy("customer_id") \
.count() \
.filter(col("count")>1) \
.show()

Products table cleaning in Silver Layer


In [0]:
products_bronze_df.count()

In [0]:
products_bronze_df.dropDuplicates().count()

In [0]:
from pyspark.sql.functions import col,sum

In [0]:
products_bronze_df.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in products_bronze_df.columns]
).show()


In [0]:
product_silver_df = products_bronze_df.dropDuplicates()

In [0]:
product_silver_df.select("category").distinct().show()

In [0]:
from pyspark.sql.functions import initcap, trim, col

product_silver_df = product_silver_df.withColumn(
    "category",
    initcap(trim(col("category")))
)

In [0]:
product_silver_df.select("category").show()

In [0]:
product_silver_df.select("category").show()


In [0]:
    product_silver_df.groupBy("product_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

create a delta table

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

In [0]:
(
    customers_silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.customers")
)

In [0]:
(
    product_silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.products")
    
)

In [0]:
display(spark.sql("SHOW TABLES IN workspace.silver"))

Orders table cleaning in silver layer

In [0]:
orders_bronze_df = spark.table("workspace.bronze.orders")

In [0]:
orders_bronze_df.count()

In [0]:
orders_bronze_df.dropDuplicates().count()

In [0]:
orders_bronze_df.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in orders_bronze_df.columns]
).show()

In [0]:
orders_silver_df = orders_bronze_df.dropDuplicates()


In [0]:
orders_silver_df.filter(col("customer_id").isNull()).show()

In [0]:
invalid_customers_df = (
    orders_silver_df
    .join(
        spark.table("workspace.silver.customers").select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
)

invalid_customers_df.show()

In [0]:
invalid_customers_df = (
    orders_silver_df
    .join(spark.table("workspace.silver.customers").select("customer_id"),
          on = "customer_id",
          how = "left_anti"
          )
)
invalid_customers_df.show()

In [0]:
invalid_product_df = (
    orders_silver_df 
    .join(
        spark.table("workspace.silver.products").select("product_id"),
        on = "product_id",
        how = "left_anti"
    )
)
invalid_product_df.show()

In [0]:
orders_silver_df = (
    orders_silver_df
    .filter(col("customer_id").isNotNull())
    .join(
        spark.table("workspace.silver.customers").select("customer_id"),
        on="customer_id",
        how="left_semi"
    )
)

In [0]:
orders_silver_df = (
    orders_silver_df
    .join(
        spark.table("workspace.silver.products").select("product_id"),
        on="product_id",
        how="left_semi"
    )
)

In [0]:
orders_silver_df.filter(col("customer_id").isNull()).count()

In [0]:
orders_silver_df.join(
    spark.table("workspace.silver.products").select("product_id"),
    on="product_id",
    how="left_anti"
).count()

In [0]:
orders_silver_df.filter(
    col("quantity") <= 0
).show()

In [0]:
orders_silver_df = orders_silver_df.filter(
    col("quantity") > 0
)

In [0]:
orders_silver_df.filter(col("quantity")<= 0).count()

In [0]:
orders_silver_df.select("order_status").distinct().show()

In [0]:
from pyspark.sql.functions import trim, initcap, col
orders_silver_df = orders_silver_df.withColumn(
    "order_status",
    initcap(trim(col("order_status")))
)

In [0]:
orders_silver_df.select("order_status").distinct().show()

In [0]:
orders_silver_df.printSchema()

In [0]:
(
    orders_silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.orders")
)


In [0]:
display(spark.sql("SHOW TABLES IN workspace.silver"))